# 02 · Aggregation: GROUP BY and HAVING (BigQuery)

**Aggregate functions** (COUNT, SUM, AVG, MIN, MAX) turn many rows into one value.
`GROUP BY` makes one row per group. `HAVING` filters groups (WHERE filters rows, before grouping).

Traps in this data: NULL amounts in `orders`, 4 exact duplicate rows in `orders`, one employee with NULL `dept_id`.

**How to run:** Start each SQL cell with `%%bigquery`. Write tables as `practice.table_name`.
Each cell prints how much data it read. `%%bigquery dry` checks the size for free.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`bigquery/practice/02_aggregation.ipynb`](../../bigquery/practice/02_aggregation.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "bigquery"))
%load_ext bq_helper

## Q1. Head count per department  ·  🟢 easy

Show `dept_id` and the number of employees `num_employees` for each department id. Keep the group for employees with no department (NULL).

*Expected result: 7 rows · columns: `dept_id`, `num_employees`*

**How it works:** `GROUP BY dept_id` makes one group per department id (NULL is its own group); `COUNT(*)` counts the rows in each group.

In [ ]:
%%bigquery
SELECT dept_id, COUNT(*) AS num_employees
FROM practice.employees
GROUP BY dept_id
ORDER BY dept_id

,dept_id,num_employees
0,<NA>,1
1,1,7
2,2,6
3,3,4
4,4,4
5,5,3
6,6,5


**Why / traps:** GROUP BY puts all NULLs into one group. ORDER BY puts NULL first in BigQuery and Spark, but **last** in Oracle (use `NULLS FIRST` / `NULLS LAST` to choose).

## Q2. Salary stats per department  ·  🟢 easy

For each `dept_id` (skip NULL) show `num_employees`, `total_salary`, `avg_salary` (rounded to 2 decimals), `min_salary`, `max_salary`.

*Expected result: 6 rows · columns: `dept_id`, `num_employees`, `total_salary`, `avg_salary`, `min_salary`, `max_salary`*

**How it works:** Remove the NULL department, group by `dept_id`, and compute all five aggregates in the same SELECT.

In [ ]:
%%bigquery
SELECT dept_id,
       COUNT(*)               AS num_employees,
       SUM(salary)            AS total_salary,
       ROUND(AVG(salary), 2)  AS avg_salary,
       MIN(salary)            AS min_salary,
       MAX(salary)            AS max_salary
FROM practice.employees
WHERE dept_id IS NOT NULL
GROUP BY dept_id
ORDER BY dept_id

,dept_id,num_employees,total_salary,avg_salary,min_salary,max_salary
0,1,7,940000.000000000,134285.710000000,95000.000000000,250000.000000000
1,2,6,545000.000000000,90833.330000000,65000.000000000,140000.000000000
2,3,4,360000.000000000,90000.000000000,60000.000000000,130000.000000000
3,4,4,431000.000000000,107750.000000000,98000.000000000,135000.000000000
4,5,3,245000.000000000,81666.670000000,68000.000000000,105000.000000000
5,6,5,400000.000000000,80000.000000000,55000.000000000,115000.000000000


## Q3. Three kinds of COUNT  ·  🟡 medium

On the `orders` table, in one row, show:
- `all_rows` = number of rows
- `rows_with_amount` = rows where amount is not NULL
- `unique_orders` = number of different order ids

*Expected result: 1 row · columns: `all_rows`, `rows_with_amount`, `unique_orders`*

**How it works:** One scan, three counts: `COUNT(*)` counts rows, `COUNT(amount)` skips NULLs, `COUNT(DISTINCT order_id)` counts different ids.

In [ ]:
%%bigquery
SELECT COUNT(*)                 AS all_rows,
       COUNT(amount)            AS rows_with_amount,
       COUNT(DISTINCT order_id) AS unique_orders
FROM practice.orders

,all_rows,rows_with_amount,unique_orders
0,108,102,104


**Why / traps:** `COUNT(*)` counts rows. `COUNT(col)` skips NULLs. `COUNT(DISTINCT col)` counts different values. Here 108 rows but only 104 orders, because of 4 duplicate rows.

## Q4. Orders per status  ·  🟢 easy

Show each `status` and how many orders `num_orders` have it. Most common first.

*Expected result: 4 rows · columns: `status`, `num_orders`*

**How it works:** Group by `status`, count each group, then sort by the count (biggest first) with `status` as tie-breaker.

In [ ]:
%%bigquery
SELECT status, COUNT(*) AS num_orders
FROM practice.orders
GROUP BY status
ORDER BY num_orders DESC, status

,status,num_orders
0,completed,73
1,cancelled,16
2,returned,11
3,pending,8


## Q5. Big departments (HAVING)  ·  🟢 easy

Show `dept_id` and `num_employees` for departments with **more than 4** employees.

*Expected result: 3 rows · columns: `dept_id`, `num_employees`*

**How it works:** Group and count first; `HAVING COUNT(*) > 4` then filters the **groups**. (WHERE can't do it: it runs before grouping.)

In [ ]:
%%bigquery
SELECT dept_id, COUNT(*) AS num_employees
FROM practice.employees
WHERE dept_id IS NOT NULL
GROUP BY dept_id
HAVING COUNT(*) > 4

,dept_id,num_employees
0,1,7
1,2,6
2,6,5


**Why / traps:** You cannot use an aggregate in WHERE: WHERE runs **before** GROUP BY. Order of running: FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY.

## Q6. Completed vs cancelled per customer  ·  🟡 medium

For each `customer_id` that has orders, show `completed_orders` and `cancelled_orders` (count of rows with that status) in **one** query.

*Expected result: 32 rows · columns: `customer_id`, `completed_orders`, `cancelled_orders`*

**How it works:** `CASE WHEN status = 'completed' THEN 1 ELSE 0 END` gives 1 for matching rows; `SUM` adds the 1s = a count per condition, all in one GROUP BY.

In [ ]:
%%bigquery
SELECT customer_id,
       SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_orders,
       SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders
FROM practice.orders
GROUP BY customer_id

,customer_id,completed_orders,cancelled_orders
0,2,0,1
1,11,1,1
2,18,3,1
3,22,2,1
4,23,1,1
5,24,2,1
6,26,5,4
7,27,6,1
8,30,4,2
9,31,3,3


**Why / traps:** This is called **conditional aggregation**. It works everywhere. Shorter versions: BigQuery `COUNTIF(status = 'completed')`, Spark `count_if(...)`. In Spark you cannot `SUM` a true/false value.

## Q7. AVG and NULL  ·  🟡 medium

In one row show `avg_ignoring_null` = AVG(amount) and `avg_null_as_zero` = average when a missing amount counts as 0. Round both to 2 decimals. Why are they different?

*Expected result: 1 row · columns: `avg_ignoring_null`, `avg_null_as_zero`*

**How it works:** `AVG(amount)` divides by rows **with** an amount. `AVG(COALESCE(amount, 0))` turns NULL into 0 first, so it divides by all rows and the result is lower.

In [ ]:
%%bigquery
SELECT ROUND(AVG(amount), 2)              AS avg_ignoring_null,
       ROUND(AVG(COALESCE(amount, 0)), 2) AS avg_null_as_zero
FROM practice.orders

,avg_ignoring_null,avg_null_as_zero
0,101.170000000,95.550000000


**Why / traps:** AVG (like SUM, MIN, MAX) **skips NULL** rows: it divides by 102, not 108. Ask the interviewer which one the business wants.

## Q8. Completed revenue per year (duplicates!)  ·  🟡 medium

Show `order_year` and `revenue` = total amount of **completed** orders per year. The orders table has exact duplicate rows: count each order only once.

*Expected result: 3 rows · columns: `order_year`, `revenue`*

**How it works:** 1) Remove exact duplicate rows (`SELECT DISTINCT *` in a subquery). 2) Keep completed orders. 3) Group by the year and SUM the amount.

In [ ]:
%%bigquery
SELECT EXTRACT(YEAR FROM order_date) AS order_year,
       SUM(amount)                    AS revenue
FROM (SELECT DISTINCT order_id, customer_id, order_date, amount, status
      FROM practice.orders) o
WHERE status = 'completed'
GROUP BY EXTRACT(YEAR FROM order_date)
ORDER BY order_year

,order_year,revenue
0,2023,517.000000000
1,2024,2771.000000000
2,2025,2827.000000000


**Why / traps:** Always check for duplicates before you SUM. Without the DISTINCT the revenue is too high. Oracle 19c cannot `GROUP BY order_year` (the alias), so repeat the expression; BigQuery and Spark allow the alias.

## Q9. Loyal customers  ·  🟡 medium

Show `customer_id` and `num_orders` for customers with **at least 5 different orders**. Most orders first, then customer_id.

*Expected result: 7 rows · columns: `customer_id`, `num_orders`*

**How it works:** Count **different** order ids per customer (duplicates must not count twice), keep groups with `>= 5`, sort.

In [ ]:
%%bigquery
SELECT customer_id, COUNT(DISTINCT order_id) AS num_orders
FROM practice.orders
GROUP BY customer_id
HAVING COUNT(DISTINCT order_id) >= 5
ORDER BY num_orders DESC, customer_id

,customer_id,num_orders
0,26,9
1,27,9
2,30,9
3,28,8
4,25,6
5,31,6
6,29,5


## Q10. Hires per department per year  ·  🟢 easy

Show `dept_id`, `hire_year`, `num_hired` (skip NULL dept_id). Sort by dept_id, hire_year.

*Expected result: 29 rows · columns: `dept_id`, `hire_year`, `num_hired`*

**How it works:** Group by two things at once (department and hire year): one output row per pair.

In [ ]:
%%bigquery
SELECT dept_id, EXTRACT(YEAR FROM hire_date) AS hire_year, COUNT(*) AS num_hired
FROM practice.employees
WHERE dept_id IS NOT NULL
GROUP BY dept_id, EXTRACT(YEAR FROM hire_date)
ORDER BY dept_id, hire_year

,dept_id,hire_year,num_hired
0,1,2015,1
1,1,2016,1
2,1,2018,1
3,1,2019,1
4,1,2020,1
5,1,2021,1
6,1,2022,1
7,2,2016,1
8,2,2017,1
9,2,2018,1


**Why / traps:** Every column in SELECT that is not inside an aggregate must be in GROUP BY.

## Q11. Categories with no cheap products  ·  🟡 medium

Show `category` and `cheapest` (the lowest price) for categories where **every** product costs at least 15.

*Expected result: 2 rows · columns: `category`, `cheapest`*

**How it works:** "All products cost ≥ 15" means the cheapest one does. So group by category, compute `MIN(price)`, and keep groups with `HAVING MIN(price) >= 15`.

In [ ]:
%%bigquery
SELECT category, MIN(price) AS cheapest
FROM practice.products
GROUP BY category
HAVING MIN(price) >= 15

,category,cheapest
0,Books,30.000000000
1,Sports,30.000000000


**Why / traps:** "Every product ≥ 15" is the same as "the cheapest product ≥ 15". This MIN/MAX trick answers many "all / none" questions.

## Q12. Subtotals with ROLLUP  ·  🔴 hard

Show `category`, `num_products` and `avg_price` (2 decimals) per category, **plus one extra row** for all products together (category = NULL).

*Expected result: 6 rows · columns: `category`, `num_products`, `avg_price`*

**How it works:** `ROLLUP(category)` makes the normal per-category groups **plus** one grand-total group where category is NULL.

In [ ]:
%%bigquery
SELECT category,
       COUNT(*)              AS num_products,
       ROUND(AVG(price), 2)  AS avg_price
FROM practice.products
GROUP BY ROLLUP (category)
ORDER BY category

,category,num_products,avg_price
0,None,20,37.700000000
1,Books,4,36.250000000
2,Clothing,4,23.750000000
3,Electronics,6,53.670000000
4,Home,4,18.000000000
5,Sports,2,60.000000000


**Why / traps:** `ROLLUP(a, b)` adds subtotals for (a) and a grand total. `CUBE` adds every combination. `GROUPING SETS` lets you list the groups yourself. All 3 engines support them.